# Lab 10 — Incremental CDC Synchronization

**WHAT:** Synchronize a Delta source with a Delta target using Change Data Feed.

**WHY:** Process only new changes instead of reloading the entire source.

**INPUT:** customers_delta, cdc_checkpoint

**OUTPUT:** customers_target, customer_change_history, updated checkpoint

**KEY CONCEPTS:** CDF, commit versions, MERGE, idempotency, SCD Type 1.

**EXPECTED RESULT:** When no new changes exist, skip processing without modifying any tables.

**RERUN SAFETY:** No table recreation. The checkpoint advances only after successful reconciliation.

In [ ]:
# WHAT: Read deployment parameters. Defaults are the Personal DEV values.
# WHY: The same notebook runs in Personal DEV and Azure PROD without edits;
#      the Azure Job passes its own catalog, schema and foreign catalog.
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "lab_10_federation_cdc")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

# Identifiers are interpolated into SQL, so only plain names are accepted.
for name, value in {"catalog": CATALOG, "schema": SCHEMA}.items():
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value), f"Invalid {name}: {value!r}"

BASE = f"{CATALOG}.{SCHEMA}"
print(f"Delta schema: {BASE}")

In [0]:
# Step 1: Identify source, target, history and checkpoint.

# BASE comes from the parameter cell above.

SOURCE = f"{BASE}.customers_delta"
TARGET = f"{BASE}.customers_target"
HISTORY = f"{BASE}.customer_change_history"
CHECKPOINT = f"{BASE}.cdc_checkpoint"

PIPELINE = "customers_delta_to_target"

# Last successfully processed Delta version
rows = spark.sql(f"""
    SELECT last_processed_version
    FROM {CHECKPOINT}
    WHERE pipeline_name = '{PIPELINE}'
""").collect()

assert len(rows) == 1, "Expected exactly one checkpoint"

last_version = int(rows[0]["last_processed_version"])

# Latest version committed to the source
latest_version = int(
    spark.sql(f"DESCRIBE HISTORY {SOURCE} LIMIT 1")
    .first()["version"]
)

assert latest_version >= last_version, (
    "Source version is older than the checkpoint"
)

start_version = last_version + 1

print(f"Last processed: {last_version}")
print(f"Latest source:  {latest_version}")

if start_version > latest_version:
    print("NO NEW CHANGES - skipping CDC processing")
else:
    print(f"New version range: {start_version} to {latest_version}")

In [0]:

event_count = 0

if start_version <= latest_version:

    # Read only new CDF events, including preimages for audit.
    events = spark.sql(f"""
        SELECT
            customer_id, first_name, last_name,
            email, city, country, signup_date,
            _change_type, _commit_version,
            _commit_timestamp
        FROM table_changes(
            '{SOURCE}', {start_version}, {latest_version}
        )
    """)

    event_count = events.count()

    assert events.filter("customer_id IS NULL").count() == 0

    # The audit key must uniquely identify each event.
    duplicates = (
        events.groupBy(
            "customer_id", "_commit_version", "_change_type"
        )
        .count()
        .filter("count > 1")
        .count()
    )
    assert duplicates == 0, "Duplicate CDF event keys"

    events.createOrReplaceTempView("lab10_cdf_events")

    if event_count > 0:

        # Latest final-state event per customer.
        spark.sql("""
            CREATE OR REPLACE TEMP VIEW lab10_latest_changes AS
            SELECT *
            FROM lab10_cdf_events
            WHERE _change_type IN (
                'insert', 'update_postimage', 'delete'
            )
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY _commit_version DESC
            ) = 1
        """)

        # Synchronize current state (SCD Type 1).
        spark.sql(f"""
            MERGE INTO {TARGET} t
            USING lab10_latest_changes s
            ON t.customer_id = s.customer_id

            WHEN MATCHED AND s._change_type = 'delete'
                THEN DELETE

            WHEN MATCHED
                AND s._change_type <> 'delete'
                AND NOT (
                    t.first_name <=> s.first_name
                    AND t.last_name <=> s.last_name
                    AND t.email <=> s.email
                    AND t.city <=> s.city
                    AND t.country <=> s.country
                    AND t.signup_date <=> s.signup_date
                )
                THEN UPDATE SET
                    t.first_name = s.first_name,
                    t.last_name = s.last_name,
                    t.email = s.email,
                    t.city = s.city,
                    t.country = s.country,
                    t.signup_date = s.signup_date

            WHEN NOT MATCHED AND s._change_type <> 'delete'
                THEN INSERT (
                    customer_id, first_name, last_name,
                    email, city, country, signup_date
                )
                VALUES (
                    s.customer_id, s.first_name, s.last_name,
                    s.email, s.city, s.country, s.signup_date
                )
        """)

        # Preserve every CDC event, including update_preimage.
        # MERGE makes audit writes safe to replay.
        spark.sql(f"""
            MERGE INTO {HISTORY} t
            USING lab10_cdf_events s
            ON t.customer_id = s.customer_id
               AND t._commit_version = s._commit_version
               AND t._change_type = s._change_type

            WHEN NOT MATCHED THEN INSERT (
                customer_id, first_name, last_name,
                email, city, country, signup_date,
                _change_type, _commit_version, _commit_timestamp
            )
            VALUES (
                s.customer_id, s.first_name, s.last_name,
                s.email, s.city, s.country, s.signup_date,
                s._change_type, s._commit_version,
                s._commit_timestamp
            )
        """)

    print(f"CDC events processed: {event_count}")

else:
    print("CDC processing skipped — checkpoint is current")


In [0]:

# Compare against the precise source version we processed.
# This prevents later source writes from affecting this check.

differences = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM (
        SELECT * FROM {SOURCE} VERSION AS OF {latest_version}
        EXCEPT ALL
        SELECT * FROM {TARGET}
    )
""").first()["total"]

reverse_differences = spark.sql(f"""
    SELECT COUNT(*) AS total
    FROM (
        SELECT * FROM {TARGET}
        EXCEPT ALL
        SELECT * FROM {SOURCE} VERSION AS OF {latest_version}
    )
""").first()["total"]

assert differences == 0, "Source-to-target mismatch"
assert reverse_differences == 0, "Target-to-source mismatch"

if event_count > 0:
    # Verify the new events were fully preserved in history.
    missing_audit = spark.sql(f"""
        SELECT COUNT(*) AS total
        FROM (
            SELECT * FROM lab10_cdf_events
            EXCEPT ALL
            SELECT *
            FROM {HISTORY}
            WHERE _commit_version BETWEEN
                {start_version} AND {latest_version}
        )
    """).first()["total"]

    assert missing_audit == 0, "Missing CDC audit events"

# Advance checkpoint only after all validations succeed.
if latest_version > last_version:
    spark.sql(f"""
        MERGE INTO {CHECKPOINT} t
        USING (
            SELECT
                '{PIPELINE}' AS pipeline_name,
                CAST({latest_version} AS BIGINT)
                    AS last_processed_version,
                current_timestamp() AS processed_at
        ) s
        ON t.pipeline_name = s.pipeline_name

        WHEN MATCHED AND
            t.last_processed_version < s.last_processed_version
        THEN UPDATE SET
            t.last_processed_version = s.last_processed_version,
            t.processed_at = s.processed_at

        WHEN NOT MATCHED THEN
            INSERT (
                pipeline_name,
                last_processed_version,
                processed_at
            )
            VALUES (
                s.pipeline_name,
                s.last_processed_version,
                s.processed_at
            )
    """)

print("PASS: Source and target match")
print(f"PASS: Latest processed version = {latest_version}")
print(f"Events in this run: {event_count}")
